In [15]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.0 MB/s eta 0:00:00


In [16]:
!pip install lightgbm -q

## IC50 Regression
IC50 — это минимальная концентрация соединения или экстракта, которая подавляет пролиферацию 50% патогенов или вирусов. То есть это показатель минимальной ингибирующей концентрации для подавления активности патогена.

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import optuna

from sklearn.model_selection import train_test_split, KFold, cross_validate, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score, classification_report

# Load the dataset
file_path = '/content/cleaned_molecular_data.csv'
df = pd.read_csv(file_path)

# Display basic information
print("Dataset Shape:", df.shape)
display(df.head())
print(df.info())

Dataset Shape: (966, 214)


,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,is_outlier
0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,...,0,0,0,0,0,0,0,3,0,1
1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,...,0,0,0,0,0,0,0,3,0,1
2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,...,0,0,0,0,0,0,0,3,0,1
3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,398.679,...,0,0,0,0,0,0,0,4,0,1
4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,466.713,...,0,0,0,0,0,0,0,0,0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Columns: 214 entries, IC50, mM to is_outlier
dtypes: float64(107), int64(107)
memory usage: 1.6 MB
None


### Feature Selection for IC50 Regression

We will perform the following steps:
1.  **Correlation Analysis**: Identify features highly correlated with the target `IC50`.
2.  **Remove Low Variance Features**: Drop features that are constant or near-constant.
3.  **Multicollinearity Check**: Identify features that are highly correlated with each other to reduce redundancy.

In [18]:
# 1. Separate Target and Features
target_col = 'IC50, mM'
exclude_cols = ['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8']

# Filter features: only numeric and not in the exclude list
relevant_features = [col for col in df.select_dtypes(include=[np.number]).columns if col not in exclude_cols]

X = df[relevant_features]
y = df[target_col]

# 2. Drop constant features (Variance Threshold)
selector = VarianceThreshold(threshold=0)
selector.fit(X)
relevant_features = X.columns[selector.get_support()].tolist()

print(f"Total features selected (excluding target-related and constant): {len(relevant_features)}")
print(f"Top 10 selected features for reference: {relevant_features[:10]}")

Total features selected (excluding target-related and constant): 193
Top 10 selected features for reference: ['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons']


### Log-transformation and Model Training

1.  **Transform Target**: Apply `log1p` to `IC50, mM` to normalize the distribution.
2.  **Split Data**: 80/20 train-test split.
3.  **Train Models**: Random Forest, Gradient Boosting, and XGBoost.
4.  **Evaluate**: Compare metrics.

### Базовые модели (стандартные параметры) для IC50
Прежде чем переходить к оптимизации, оценим качество моделей "из коробки".

In [19]:
# 1. Prepare Data using the updated feature list
X_selected = df[relevant_features]
y_log = -np.log10(df['IC50, mM'])

X_train, X_test, y_train, y_test = train_test_split(X_selected, y_log, test_size=0.2, random_state=42)

# 2. Define Models
models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# 3. Train and Evaluate
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results[name] = {"MSE": mse, "R2": r2}

# Display Results
results_df = pd.DataFrame(results).T
display(results_df)

,MSE,R2
Random Forest,0.574406,0.392425
Gradient Boosting,0.493098,0.478429
XGBoost,0.556995,0.410842


In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'random_state': 42
    }

    model = GradientBoostingRegressor(**params)
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    cv_results = cross_validate(model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error', return_train_score=True)

    avg_val_rmse = np.mean(np.sqrt(-cv_results['test_score']))
    avg_train_rmse = np.mean(np.sqrt(-cv_results['train_score']))

    gap = abs(avg_val_rmse - avg_train_rmse)
    penalty_factor = 2

    return avg_val_rmse + penalty_factor * gap

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("Best parameters:", study.best_params)
print("Best penalized RMSE:", study.best_value)

In [ ]:
best_params_gbr=study.best_params

In [25]:
best_params_gbr={'n_estimators': 95,
 'learning_rate': 0.009954759857324386,
 'max_depth': 3,
 'min_samples_split': 15,
 'min_samples_leaf': 4,
 'subsample': 0.6688874501352873}

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 15),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42
    }

    model = RandomForestRegressor(**params)
    kf = KFold(n_splits=4, shuffle=True, random_state=42)
    cv_results = cross_validate(model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error', return_train_score=True)

    avg_val_rmse = np.mean(np.sqrt(-cv_results['test_score']))
    avg_train_rmse = np.mean(np.sqrt(-cv_results['train_score']))
    gap = abs(avg_val_rmse - avg_train_rmse)

    return avg_val_rmse + 2 * gap

study_rf = optuna.create_study(direction='minimize')
study_rf.optimize(objective_rf, n_trials=50)

print("Best RF parameters:", study_rf.best_params)
print("Best RF penalized RMSE:", study_rf.best_value)

In [13]:
best_params_rf=study_rf.best_params

In [21]:
best_params_rf={'random_state': 42,
 'n_estimators': 50,
 'max_depth': 5,
 'min_samples_split': 8,
 'min_samples_leaf': 13,
 'max_features': 'log2'}

In [ ]:
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'verbosity': -1
    }

    model = lgb.LGBMRegressor(**params)
    kf = KFold(n_splits=4, shuffle=True, random_state=42)
    cv_results = cross_validate(model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error', return_train_score=True)

    avg_val_rmse = np.mean(np.sqrt(-cv_results['test_score']))
    avg_train_rmse = np.mean(np.sqrt(-cv_results['train_score']))
    gap = abs(avg_val_rmse - avg_train_rmse)

    return avg_val_rmse + 2 * gap

study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=50)

print("Best LGBM parameters:", study_lgbm.best_params)
print("Best LGBM penalized RMSE:", study_lgbm.best_value)

In [15]:
best_params_lgbm=study_lgbm.best_params

In [22]:
best_params_lgbm={'random_state': 42,
 'verbosity': -1,
 'n_estimators': 79,
 'learning_rate': 0.01310624327085752,
 'num_leaves': 22,
 'max_depth': 3,
 'min_child_samples': 39,
 'subsample': 0.7158940235204034,
 'colsample_bytree': 0.9974001880706533}

In [29]:
final_results = []

models_to_test = {
    "GBR": GradientBoostingRegressor(**best_params_gbr ),
    "RF": RandomForestRegressor(**best_params_rf),
    "LGBM": lgb.LGBMRegressor(**best_params_lgbm)
}

for name, model in models_to_test.items():
    model.fit(X_train, y_train)
    tr_p = model.predict(X_train)
    te_p = model.predict(X_test)

    tr_rmse = np.sqrt(mean_squared_error(y_train, tr_p))
    te_rmse = np.sqrt(mean_squared_error(y_test, te_p))
    te_r2 = r2_score(y_test, te_p)

    final_results.append({
        "Model": name,
        "Train RMSE": tr_rmse,
        "Test RMSE": te_rmse,
        "Test R2": te_r2,
        "Gap": abs(tr_rmse - te_rmse)
    })

comparison_df = pd.DataFrame(final_results)
display(comparison_df)

,Model,Train RMSE,Test RMSE,Test R2,Gap
0,GBR,0.731772,0.801295,0.320850,0.069522
1,RF,0.662618,0.775074,0.364571,0.112456
2,LGBM,0.728693,0.775871,0.363263,0.047178


In [16]:
# Evaluate and compare all models
final_results = []

models_to_test = {
    "GBR": GradientBoostingRegressor(**study.best_params, random_state=42),
    "RF": RandomForestRegressor(**study_rf.best_params, random_state=42),
    "LGBM": lgb.LGBMRegressor(**study_lgbm.best_params, random_state=42, verbosity=-1)
}

for name, model in models_to_test.items():
    model.fit(X_train, y_train)
    tr_p = model.predict(X_train)
    te_p = model.predict(X_test)

    tr_rmse = np.sqrt(mean_squared_error(y_train, tr_p))
    te_rmse = np.sqrt(mean_squared_error(y_test, te_p))
    te_r2 = r2_score(y_test, te_p)

    final_results.append({
        "Model": name,
        "Train RMSE": tr_rmse,
        "Test RMSE": te_rmse,
        "Test R2": te_r2,
        "Gap": abs(tr_rmse - te_rmse)
    })

comparison_df = pd.DataFrame(final_results)
display(comparison_df)

,Model,Train RMSE,Test RMSE,Test R2,Gap
0,GBR,0.733647,0.804831,0.314842,0.071184
1,RF,0.649699,0.770564,0.371944,0.120865
2,LGBM,0.668859,0.745846,0.411591,0.076987


In [30]:
#best_gb = GradientBoostingRegressor(**study.best_params, random_state=42)
best_gb = GradientBoostingRegressor(**best_params_gbr)
best_gb.fit(X_train, y_train)

train_preds = best_gb.predict(X_train)
test_preds = best_gb.predict(X_test)

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
train_r2 = r2_score(y_train, train_preds)
test_r2 = r2_score(y_test, test_preds)

print(f"Final Model Results (Optimized for RMSE):")
print(f"---")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE:  {test_rmse:.4f}")
print(f"RMSE Gap:   {abs(train_rmse - test_rmse):.4f}")
print(f"---")
print(f"Train R2:   {train_r2:.4f}")
print(f"Test R2:    {test_r2:.4f}")
print(f"R2 Gap:     {abs(train_r2 - test_r2):.4f}")

Final Model Results (Optimized for RMSE):
---
Train RMSE: 0.7365
Test RMSE:  0.8077
RMSE Gap:   0.0712
---
Train R2:   0.3634
Test R2:    0.3100
R2 Gap:     0.0534


In [31]:
relevant_features

['MaxAbsEStateIndex',
 'MaxEStateIndex',
 'MinAbsEStateIndex',
 'MinEStateIndex',
 'qed',
 'SPS',
 'MolWt',
 'HeavyAtomMolWt',
 'ExactMolWt',
 'NumValenceElectrons',
 'MaxPartialCharge',
 'MinPartialCharge',
 'MaxAbsPartialCharge',
 'MinAbsPartialCharge',
 'FpDensityMorgan1',
 'FpDensityMorgan2',
 'FpDensityMorgan3',
 'BCUT2D_MWHI',
 'BCUT2D_MWLOW',
 'BCUT2D_CHGHI',
 'BCUT2D_CHGLO',
 'BCUT2D_LOGPHI',
 'BCUT2D_LOGPLOW',
 'BCUT2D_MRHI',
 'BCUT2D_MRLOW',
 'AvgIpc',
 'BalabanJ',
 'BertzCT',
 'Chi0',
 'Chi0n',
 'Chi0v',
 'Chi1',
 'Chi1n',
 'Chi1v',
 'Chi2n',
 'Chi2v',
 'Chi3n',
 'Chi3v',
 'Chi4n',
 'Chi4v',
 'HallKierAlpha',
 'Ipc',
 'Kappa1',
 'Kappa2',
 'Kappa3',
 'LabuteASA',
 'PEOE_VSA1',
 'PEOE_VSA10',
 'PEOE_VSA11',
 'PEOE_VSA12',
 'PEOE_VSA13',
 'PEOE_VSA14',
 'PEOE_VSA2',
 'PEOE_VSA3',
 'PEOE_VSA4',
 'PEOE_VSA5',
 'PEOE_VSA6',
 'PEOE_VSA7',
 'PEOE_VSA8',
 'PEOE_VSA9',
 'SMR_VSA1',
 'SMR_VSA10',
 'SMR_VSA2',
 'SMR_VSA3',
 'SMR_VSA4',
 'SMR_VSA5',
 'SMR_VSA6',
 'SMR_VSA7',
 'SMR_VSA9'

In [32]:
df['is_outlier'].value_counts()

,count
is_outlier,
1,917
-1,49


In [33]:

target_col = 'IC50, mM'
exclude_cols = ['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8','is_outlier']
relevant_features = [col for col in df.select_dtypes(include=[np.number]).columns if col not in exclude_cols]

df_filtered = df[df['is_outlier'] == 1].copy()

X_filtered = df_filtered[relevant_features]
y_log_filtered = -np.log10(df_filtered['IC50, mM'])

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_filtered, y_log_filtered, test_size=0.2, random_state=42)

print(f"Original samples: {len(df)}")
print(f"Filtered samples: {len(df_filtered)}")
print(f"Outliers removed: {len(df) - len(df_filtered)}")

Original samples: 966
Filtered samples: 917
Outliers removed: 49


In [35]:
# Re-evaluate models on filtered data using previously found best parameters
filtered_results = []
best_params={ 'random_state':42,'n_estimators': 152, 'learning_rate': 0.006029183608657581, 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 20, 'subsample': 0.6982703824807429}
best_params_rf={'random_state':42,'n_estimators': 50, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 13, 'max_features': 'log2'}
best_params_lgbm={'random_state':42,'verbosity':-1,'n_estimators': 79, 'learning_rate': 0.01310624327085752, 'num_leaves': 22, 'max_depth': 3, 'min_child_samples': 39, 'subsample': 0.7158940235204034, 'colsample_bytree': 0.9974001880706533}
models_to_test_f = {
    "GBR (Filtered)": GradientBoostingRegressor(**best_params),
    "RF (Filtered)": RandomForestRegressor(**best_params_rf),
    "LGBM (Filtered)": lgb.LGBMRegressor(**best_params_lgbm)
}

for name, model in models_to_test_f.items():
    model.fit(X_train_f, y_train_f)
    tr_p = model.predict(X_train_f)
    te_p = model.predict(X_test_f)

    tr_rmse = np.sqrt(mean_squared_error(y_train_f, tr_p))
    te_rmse = np.sqrt(mean_squared_error(y_test_f, te_p))
    te_r2 = r2_score(y_test_f, te_p)

    filtered_results.append({
        "Model": name,
        "Train RMSE": tr_rmse,
        "Test RMSE": te_rmse,
        "Test R2": te_r2,
        "Gap": abs(tr_rmse - te_rmse)
    })

comparison_filtered_df = pd.DataFrame(filtered_results)
display(comparison_filtered_df)

,Model,Train RMSE,Test RMSE,Test R2,Gap
0,GBR (Filtered),0.747028,0.712949,0.353701,0.034078
1,RF (Filtered),0.688806,0.668066,0.432515,0.020740
2,LGBM (Filtered),0.741016,0.710352,0.358402,0.030664


видно что фильтрация выбросов сильно улучшает показатели качества